# 68 vs 81 niños en la evaluación encadenada (sin GPU)
De los niños que están en ambas bases (ICDAS v4 y corpus de segmentación), ¿por qué algunos no entran en la evaluación encadenada?
- **(a)** no tiene ninguna foto en común entre las dos bases;
- **(b)** comparte foto, pero ningún recorte tiene FDI legible en el centro de la máscara de referencia;
- **(c)** otro motivo.

Además: dónde quedan los recortes C de esos niños (y de toda la base), para explicar por qué la evaluación tiene solo 38 C.
Criterio de la evaluación encadenada (igual que `pipeline-icdas-v4`): `seg_mismo_fold & fdi.notna()`.

In [ ]:
import os, glob, zipfile, numpy as np, pandas as pd
IN, W = '/kaggle/input', '/kaggle/working'
find = lambda n: (glob.glob(f'{IN}/**/{n}', recursive=True) or [None])[0]
pd.set_option('display.width', 250)
z = zipfile.ZipFile(find('icdas_v4.zip')); man = pd.read_csv(z.open([n for n in z.namelist() if n.endswith('manifest.csv')][0]), dtype={'micdas': str, 'crop_id': str})
seg = pd.read_csv(find('segmentation_manifest.csv'), dtype={'image_id': str})
man['seg_image_id'] = man.seg_image_id.astype('string')
print('ICDAS v4:', len(man), 'recortes,', man.child_id.nunique(), 'niños | segmentación:', len(seg), 'fotos,', seg.child_id.nunique(), 'niños')
print('columnas seg:', list(seg.columns))
ev = man[man.seg_mismo_fold.fillna(False).astype(bool) & man.fdi.notna()]
print('evaluación encadenada:', len(ev), 'recortes |', ev.child_id.nunique(), 'niños |', ev.seg_image_id.nunique(), 'fotos |', ev.micdas.value_counts().reindex(['0', 'A', 'B', 'C']).to_dict())

In [ ]:
both = sorted(set(man.child_id) & set(seg.child_id))
fold_i = man.groupby('child_id').fold.first(); fold_s = seg.groupby('child_id').fold.first()
print('niños en ambas bases:', len(both), '| con el mismo fold:', int((fold_i[both] == fold_s[both]).sum()))
seg_imgs = seg.groupby('child_id').ahmed_image.apply(set); seg_ids = set(seg.image_id)
rows = []
for c in both:
    m = man[man.child_id == c]; si = seg_imgs[c]
    shared_by_img = set(m.ahmed_image) & si
    shared_by_id = set(m.seg_image_id.dropna()) & seg_ids
    in_ev = c in set(ev.child_id)
    if in_ev: mot = 'en la evaluación'
    elif not shared_by_img and not shared_by_id: mot = '(a) sin foto en común'
    elif m[m.seg_image_id.notna()].fdi.notna().sum() == 0 and (fold_i[c] == fold_s[c]): mot = '(b) foto en común, sin FDI legible'
    else: mot = '(c) otro'
    rows.append(dict(child_id=c, motivo=mot, fold_icdas=fold_i[c], fold_seg=fold_s[c], recortes=len(m), fotos_icdas=m.ahmed_image.nunique(),
                     fotos_seg=len(si), fotos_comunes_por_nombre=len(shared_by_img), recortes_con_seg_image_id=int(m.seg_image_id.notna().sum()),
                     recortes_con_fdi=int(m.fdi.notna().sum()), pureza_fdi_max=m.fdi_pureza.max(),
                     vistas_icdas='/'.join(sorted(m.view.astype(str).unique())), vistas_seg='/'.join(sorted(seg[seg.child_id == c].view.astype(str).unique())),
                     **{f'n{k}': int((m.micdas == k).sum()) for k in ['0', 'A', 'B', 'C']}))
N = pd.DataFrame(rows)
print(N.motivo.value_counts().to_string())
print(N[N.motivo != 'en la evaluación'].drop(columns=['fold_icdas', 'fold_seg']).to_string(index=False))
N.to_csv(f'{W}/ninos_ambas_bases_motivo.csv', index=False)

In [ ]:
# ---- motivo por recorte (toda la base) y recortes C ----
inboth = man.child_id.isin(both); shared_img = man.apply(lambda r: r.child_id in seg_imgs.index and r.ahmed_image in seg_imgs[r.child_id], axis=1)
man['motivo_recorte'] = np.select(
    [man.crop_id.isin(ev.crop_id), ~inboth, ~shared_img & man.seg_image_id.isna(), man.seg_image_id.notna() & man.fdi.isna()],
    ['en la evaluación', 'niño solo en ICDAS', 'foto no está en la segmentación', 'foto en común, FDI no legible en el centro'], 'otro')
T = pd.crosstab(man.motivo_recorte, man.micdas, margins=True, margins_name='Total').reindex(columns=['0', 'A', 'B', 'C', 'Total'])
print(T.to_string())
T.to_csv(f'{W}/recortes_por_motivo_y_clase.csv')
C81 = man[inboth & (man.micdas == 'C')]
print('\nRecortes C de los', len(both), 'niños en ambas bases:', len(C81), '→', C81.motivo_recorte.value_counts().to_dict())
print('Recortes C de los niños de ambas bases por vista y motivo:'); print(pd.crosstab(C81.view, C81.motivo_recorte).to_string())
b = man[man.motivo_recorte == 'foto en común, FDI no legible en el centro']
if len(b): print('\nPureza FDI en los recortes con foto en común y FDI no legible (cuantiles):', b.fdi_pureza.describe().round(2).to_dict())
oth = man[man.motivo_recorte == 'otro']
if len(oth): print('\nRecortes "otro":'); print(oth[['crop_id', 'child_id', 'ahmed_image', 'seg_image_id', 'fdi', 'seg_mismo_fold', 'micdas']].head(20).to_string(index=False))

In [ ]:
print('=== Resumen ===')
print(f"Niños en ambas bases: {len(both)} | en la evaluación: {(N.motivo == 'en la evaluación').sum()} | fuera: {(N.motivo != 'en la evaluación').sum()} →",
      N[N.motivo != 'en la evaluación'].motivo.value_counts().to_dict())
print(f"C en toda la base: {(man.micdas == 'C').sum()} | C en la evaluación: {(ev.micdas == 'C').sum()} | C de niños en ambas bases: {len(C81)} →", C81.motivo_recorte.value_counts().to_dict())
print(sorted(os.listdir(W)))